In [15]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
import spiceypy as spy
from scipy.optimize import newton
from astropy.time import Time

In [47]:
deg = np.pi/180

e_apophis = 0.1911663355386932 
a_apophis = 0.9223803173917017 * pc.constantes.au
q_apophis = 0.7460522521429133 * pc.constantes.au
i_apophis = 3.340958441017069 * deg
node_apophis = 203.8996515621043 * deg
peri_apophis = 126.6728325163065 * deg
tp = 2461042.918242006079 * 86400 # JD in seconds
mu = pc.constantes.mu_sun + pc.constantes.mu_earth + pc.constantes.mu_jupiter + pc.constantes.mu_saturn

h_apophis = np.sqrt(mu*a_apophis*(1-e_apophis**2))
b_apophis = a_apophis*np.sqrt(1-e_apophis**2)
h_apophis, b_apophis

t = Time("2029-04-13 18:52:00", scale = "tdb").jd *86400

In [48]:
t, tp

(np.float64(212737560720.0), 212634108136.1093)

In [49]:
def kepler(E):
    f = E - e_apophis*np.sin(E) - h_apophis/(a_apophis*b_apophis) * (t - tp)
    return f

E = newton(kepler , 0)
E

np.float64(23.09488863649747)

### Fórmulas

In [50]:
n = h_apophis/(a_apophis*b_apophis)
M = n*(t-tp) 

E = M + e_apophis*np.sin(M)
print(f"Anomalía excéntrica a primer orden: {E}")
E = M + e_apophis*np.sin(M) + e_apophis**2/2 * np.sin(2*M)
print(f"Anomalía excéntrica a segundo orden: {E}")
E = M + (e_apophis-1/3*e_apophis**3)*np.sin(M) + e_apophis**2/2 * np.sin(2*M) + 3/8*e_apophis**3*np.sin(3*M)
print(f"Anomalía excéntrica a tercer orden: {E}")

Anomalía excéntrica a primer orden: 23.082748237101853
Anomalía excéntrica a segundo orden: 23.092955614711627
Anomalía excéntrica a tercer orden: 23.096833560863274


In [51]:
f = 2*np.arctan(np.sqrt((1+e_apophis)/(1-e_apophis))*np.tan(E/2))

p = a_apophis*(1-e_apophis**2)
r = p / (1 + e_apophis * np.cos(f))
xf = r * np.cos(f)
yf = r * np.sin(f)
zf = 0

R = spy.eul2m(-node_apophis, -i_apophis, -peri_apophis, 3, 1, 3)

r = R @ np.array([xf, yf, zf])
r # Hay un error acá :(

array([-1.36356982e+11, -6.20646114e+10,  8.75562235e+07])

In [52]:
# Rutinas spiceypy

spy.conics([q_apophis, e_apophis, i_apophis, node_apophis, peri_apophis, M, tp, mu], t)

array([-9.63264391e+10,  1.28834098e+11, -9.15424051e+09, -2.23600785e+04,
       -1.36526866e+04,  1.99834823e+02])